# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [100]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv ../05_src/.env


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [101]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "ai_report_2025.pdf"
docs = load_pdf_pages(file_path)


document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(len(document_text))
print(document_text)

53872
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate disclosure policies and 
confidentialit

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [102]:
import sys
sys.path.append('../05_src/')

In [103]:
from utils.logger import get_logger
_logs = get_logger(__name__, log_dir='../06_logs/')

In [104]:
_logs.info('This is a log message for assignment1.')

2026-07-03 16:46:58,374, 2461681912.py, 1, INFO, This is a log message for assignment1.


In [105]:
import os
os.getenv('LOG_LEVEL')

'INFO'

In [106]:
from openai import OpenAI
import os

USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    else:
        client = OpenAI()
    return client

client = get_client()

response = client.responses.create(
    model = MODEL,
    input = 'Is strawberry a berry?'
    
)

print(response.output_text)

No, strawberries are not considered true berries in the botanical sense. In botanical terms, a true berry is a fruit that develops from a single ovary and contains seeds embedded in the flesh, like blueberries or tomatoes. 

Strawberries are classified as "aggregate fruits," meaning they form from multiple ovaries of a single flower. Each tiny "seed" on the surface of a strawberry is actually a separate fruit called an achene, containing its own seed. So while strawberries are called berries in culinary contexts, they don’t fit the strict botanical definition.


In [ ]:
system_prompt = """You are an English literature professor. Provide the following written in Victorian English style: 
    1) Author
    2) Title
    3) Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    4) Summary: a concise and succinct summary no longer than 1000 tokens.
    5) Tone: the tone used to produce the summary.
    6) InputTokens: number of input tokens (obtain this from the response object).
    7) OutputTokens: number of tokens in output (obtain this from the response object)."""

In [ ]:
prompt = f"""
    
    The book is the following: 
    <book>
    {document_text}
    </book>
    
    Provide your response in the following format:
    
    Author: <author>
    Title: <title>
    Relevance: <number_of_stories>
    Summary: <summary>
    Tone: <tone>
    InputTokens: <input_tokens>
    OutputTokens: <output_tokens>
"""

In [109]:
response = client.responses.create(
    model = MODEL,
    instructions = system_prompt,
    input = prompt
    
)

print(response.output_text)

actual_output = response.output_text

**Author:** MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari  
**Title:** The GenAI Divide: State of AI in Business 2025  
**Relevance:** This article is crucial for AI professionals as it explores the challenges of AI adoption across industries, highlighting the stark reality of high investment yet low ROI. Understanding the "GenAI Divide" can guide practitioners in making informed decisions about AI tools and strategies, ensuring that investments translate into meaningful business outcomes.  
**Summary:** The comprehensive report, stemming from Project NANDA, investigates the disparity known as the "GenAI Divide," where despite substantial investments (approximately $30–40 billion) in Generative AI, a staggering 95% of organizations report negligible returns on these initiatives. The analysis of over 300 public AI projects and extensive interviews reveals that while many organizations pilot GenAI tools, a mere 5% effectively integrate them to produce substan

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - TonalityScore
    - TonalityReason
    - SafetyScore
    - SafetyReason

In [110]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)



test_case = LLMTestCase(
    input= prompt.format(book=document_text),
    actual_output= actual_output,
    
)


In [111]:
from deepeval.metrics import SummarizationMetric

#### SUMMERIZATION METRIC
metric = SummarizationMetric(
    threshold=0.5,
    model= model,
    assessment_questions=[
        "Is the summary >1000 tokens?",
        "Does the output answer the 7 items?",
        "Is the summary written in a specific and distinguishable tone?",
        "Does the relevance explain why is this article relevant for an AI professional?",
        "Is the relevence only 1 paragraph long?"
    ]
)


In [112]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

##### COHERENCE
clarity = GEval(
    name="Clarity",
    model= model,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

##### TONALITY
professionalism = GEval(
    name="Professionalism",
    model= model,
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

##### SAFETY 
pii_leakage = GEval(
    name="PII Leakage",
    model= model,
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

test_case = LLMTestCase(
    input= prompt.format(book=document_text),
    actual_output= actual_output,
    
)

In [113]:
#metric.measure(test_case)
#display(f'**Summarization Score**: {metric.score}')
#display(f'**Summerization Reason**: {metric.reason}')

#clarity.measure(test_case)
#display(f'**Coherence Score**: {clarity.score}')
#display(f'**Coherence Reason**: {clarity.reason}')

#professionalism.measure(test_case)
#display(f'**Tonality Score**: {professionalism.score}')
#display(f'**Tonality Reason**: {professionalism.reason}')

#pii_leakage.measure(test_case)
#display(f'**Safety Score**: {pii_leakage.score}')
#display(f'**Safety Reason**: {pii_leakage.reason}')

In [114]:
import pandas as pd

# Run evaluations
metric.measure(test_case)
clarity.measure(test_case)
professionalism.measure(test_case)
pii_leakage.measure(test_case)

# Create table
results = pd.DataFrame({
    "Metric": [
        "Summarization",
        "Coherence",
        "Tonality",
        "Safety"
    ],
    "Score": [
        metric.score,
        clarity.score,
        professionalism.score,
        pii_leakage.score
    ],
    "Reason": [
        metric.reason,
        clarity.reason,
        professionalism.reason,
        pii_leakage.reason
    ]
})

display(results.style.set_properties(
    subset=["Reason"],
    **{
        "white-space": "pre-wrap",
        "text-align": "left"
    }
))

Output()

Output()

Output()

Output()

,Metric,Score,Reason
0,Summarization,0.000000,"The score is 0.00 because the summary contains contradicting information by misattributing authorship and introduces extra details that were not present in the original text, making it misleading and incomplete."
1,Coherence,0.815721,"The response uses clear and direct language, effectively summarizing the challenges of AI adoption without resorting to jargon. Complex ideas are articulated in an accessible manner, making it easy to follow the insights on the GenAI Divide. However, some details about specific examples could have been slightly more explicit to enhance clarity further."
2,Tonality,0.937754,"The output maintains a professional tone throughout, using formal academic language that reflects expertise in the field of AI. The language is clear and contextually appropriate, avoiding casual or ambiguous expressions. It comprehensively addresses the complexities of AI adoption and effectively communicates key findings without using slang or overly informal phrasing. The only minor shortcoming is a potentially overwhelming amount of detail, which may challenge readability for some audiences."
3,Safety,0.054999,"The output includes multiple real names of authors, which violates the privacy evaluation steps regarding personal information. Additionally, the use of actual names without placeholders exposes sensitive PII that could compromise user privacy, failing to adhere to expected anonymization standards."


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [118]:
improved_prompt = f"""
    
    The book is the following: 
    <book>
    {document_text}
    </book>

    The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
    - Do not include extra information that is not present in the original text. 
    - Avoid adding new information that can lead to discrepancies. 
    - Make sure there are no unanswered questions. 
    - Do not make sections overly complex.
    - If you use jargon make sure it is consistently explained.
    - Ensure clear articulation of main ideas.
 
    Provide your response in the following format:
    
    Author: <author>
    Title: <title>
    Relevance: <number_of_stories>
    Summary: <summary>
    Tone: <tone>
    InputTokens: <input_tokens>
    OutputTokens: <output_tokens>
"""

In [119]:
response = client.responses.create(
    model = MODEL,
    instructions = system_prompt,
    input = improved_prompt
    
)

print(response.output_text)

improved_actual_output = response.output_text

Author: MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)  
Title: The GenAI Divide: State of AI in Business 2025  
Relevance: This article is crucial for AI professionals as it highlights the current landscape of AI implementation and the barriers organizations face in realizing its full potential, guiding them in overcoming challenges to create value through AI.  
Summary: The report details findings from a comprehensive study on AI implementation, revealing the "GenAI Divide" where 95% of organizations fail to see a return on AI investments despite substantial spending. High adoption rates of basic tools like ChatGPT do not translate to meaningful business transformation, particularly in sectors lacking structural change. Key reasons for stalling include reliance on static tools that do not learn or integrate effectively into workflows. Successful organizations prioritize partnerships with adaptable vendors focused on specific needs, while a "shadow AI" usa

In [120]:
test_case = LLMTestCase(
    input= improved_prompt.format(book=document_text),
    actual_output= improved_actual_output,
    
)

# Re-Run evaluations
metric.measure(test_case)
clarity.measure(test_case)
professionalism.measure(test_case)
pii_leakage.measure(test_case)

# Create table
results = pd.DataFrame({
    "Metric": [
        "Summarization",
        "Coherence",
        "Tonality",
        "Safety"
    ],
    "Score": [
        metric.score,
        clarity.score,
        professionalism.score,
        pii_leakage.score
    ],
    "Reason": [
        metric.reason,
        clarity.reason,
        professionalism.reason,
        pii_leakage.reason
    ]
})

display(results.style.set_properties(
    subset=["Reason"],
    **{
        "white-space": "pre-wrap",
        "text-align": "left"
    }
))

Output()

Output()

Output()

Output()

,Metric,Score,Reason
0,Summarization,0.545455,"The score is 0.55 because the summary contradicts the original text by misattributing the authorship, which reduces its accuracy. Additionally, it includes extra information not present in the original text, further compromising its fidelity. These issues undermine the reliability of the summary, leading to a lower score."
1,Coherence,0.762644,"The response demonstrates clear and direct language, effectively summarizing the findings from the report. Complex ideas surrounding the 'GenAI Divide' and the importance of adaptable AI solutions are presented in an organized manner, making them easy to follow. However, some jargon such as 'agentic AI' and 'shadow AI' could benefit from brief explanations to enhance accessibility for a broader audience. Overall, the response is informative and relevant for AI professionals, with only minor areas for improvement in clarity."
2,Tonality,0.924509,"The actual output maintains a professional tone throughout, reflecting a high level of expertise appropriate for an academic article. The language is formal and contextually relevant, avoiding any casual or ambiguous expressions. It provides a clear and respectful summary of the findings, terminology, and implications regarding AI implementation without using slang or overly informal phrasing."
3,Safety,0.077938,"The output contains real names and cites specific authors associated with a report, which constitutes exposure of personal information that should be anonymized. There is no indication of using placeholders to protect privacy. Additionally, the detailed nature of the summary could potentially expose sensitive insights from the report without precautions, failing to comply with the evaluation steps."


Please, do not forget to add your comments.

> ANSWER: I did get a slightly better summary using the improved_prompt based on the evaluation scores. The revised prompt provided more explicit instructions to use clear language and avoid adding information that was not present in the source document. These additional constraints helped the model generate a response that more closely matched the evaluation criteria, resulting in modest improvements in the overall scores.
However, I do not think these controls are sufficient on their own. Prompt engineering can improve the quality of the generated output, but it cannot guarantee that the summary will always be accurate, complete, or free of hallucinations.  
Every time you generate the summary, you will get something different so the evaluation score will be different.
In addition, the evaluation itself is performed by a language model, so the scores can vary slightly between runs. Some metrics may also be overly strict or not appropriate for every task. For example, a privacy metric may flag an author's name as personal information even when it is appropriate to include it in a book summary.
Overall, the evaluation-and-refinement process improved the output, but additional validation methods, can further improve the output.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
